In [1]:
import subprocess, sys

# Install required packages silently
pkgs = ["requests", "pandas", "openpyxl", "tqdm"]
for pkg in pkgs:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True
    )

import requests
import pandas as pd
import os
import time
import json
from datetime import datetime

# ── Create folder structure ──────────────────────────────────
for folder in ["../data/raw", "../data/clean", "../outputs", "../notebooks"]:
    os.makedirs(folder, exist_ok=True)

print("✅  Packages installed")
print("✅  Folder structure ready:")
print(f"     {os.path.abspath('../data/raw')}")
print(f"     {os.path.abspath('../data/clean')}")
print(f"     {os.path.abspath('../outputs')}")
print()

# ── API Configuration ────────────────────────────────────────
BASE_URL    = "https://api.data.gov.my/opendosm"
RAW_DIR     = "../data/raw"
REQUEST_DELAY = 0.3   # seconds between requests (polite rate limit)

# ── Download log (tracks what was downloaded this session) ───
download_log = {}

# ============================================================
# FUNCTION 1: fetch_dosm
# For small/medium datasets (< 1000 rows)
# ============================================================
def fetch_dosm(dataset_id, limit=1000, extra_params=None, force=False):
    """
    Fetch ONE page from OpenDOSM API.
    Use for datasets with < 1000 rows.
    Use fetch_all_pages() for larger datasets.

    Parameters:
        dataset_id   : str  — OpenDOSM dataset ID
        limit        : int  — max rows per request (default 1000)
        extra_params : dict — additional query params (optional)
        force        : bool — re-download even if CSV already exists

    Returns: pd.DataFrame or None
    """
    csv_path = f"{RAW_DIR}/{dataset_id}.csv"

    # Skip if already downloaded (unless force=True)
    if not force and os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"  ⏩  {dataset_id:<42} {len(df):>6,} rows  (loaded from cache)")
        download_log[dataset_id] = {"rows": len(df), "status": "cached", "cols": list(df.columns)}
        return df

    params = {"id": dataset_id, "limit": limit}
    if extra_params:
        params.update(extra_params)

    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        time.sleep(REQUEST_DELAY)

        if response.status_code == 200:
            data = response.json()

            if isinstance(data, dict) and "data" in data:
                df = pd.DataFrame(data["data"])
            elif isinstance(data, list):
                df = pd.DataFrame(data)
            else:
                print(f"  ⚠️   {dataset_id:<42} Unexpected response format")
                download_log[dataset_id] = {"rows": 0, "status": "format_error"}
                return None

            if len(df) == 0:
                print(f"  ⚠️   {dataset_id:<42} 0 rows returned — check dataset ID")
                download_log[dataset_id] = {"rows": 0, "status": "empty"}
                return None

            df.to_csv(csv_path, index=False)
            print(f"  ✅  {dataset_id:<42} {len(df):>6,} rows  →  saved")
            download_log[dataset_id] = {"rows": len(df), "status": "ok", "cols": list(df.columns)}
            return df

        elif response.status_code == 404:
            print(f"  ❌  {dataset_id:<42} 404 Not Found — dataset ID may be wrong")
            download_log[dataset_id] = {"rows": 0, "status": "404"}
            return None
        else:
            print(f"  ❌  {dataset_id:<42} HTTP {response.status_code}")
            download_log[dataset_id] = {"rows": 0, "status": f"http_{response.status_code}"}
            return None

    except requests.exceptions.ConnectionError:
        print(f"  ❌  {dataset_id:<42} No internet — check connection")
        download_log[dataset_id] = {"rows": 0, "status": "no_internet"}
        return None
    except requests.exceptions.Timeout:
        print(f"  ❌  {dataset_id:<42} Timeout — try again later")
        download_log[dataset_id] = {"rows": 0, "status": "timeout"}
        return None
    except Exception as e:
        print(f"  ❌  {dataset_id:<42} Error: {e}")
        download_log[dataset_id] = {"rows": 0, "status": f"error: {e}"}
        return None


# ============================================================
# FUNCTION 2: fetch_all_pages
# For LARGE datasets that exceed 1000 rows (uses pagination)
# ============================================================
def fetch_all_pages(dataset_id, force=False):
    """
    Paginated fetch for large datasets.
    Automatically loops through all pages until no data left.

    Use for: population_malaysia, population_state,
             population_district, fertility_state,
             cpi_headline, cpi_headline_inflation,
             cpi_state, cpi_lowincome

    Parameters:
        dataset_id : str  — OpenDOSM dataset ID
        force      : bool — re-download even if CSV already exists

    Returns: pd.DataFrame or None
    """
    csv_path = f"{RAW_DIR}/{dataset_id}.csv"

    # Skip if already downloaded (unless force=True)
    if not force and os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"  ⏩  {dataset_id:<42} {len(df):>6,} rows  (loaded from cache)")
        download_log[dataset_id] = {"rows": len(df), "status": "cached", "cols": list(df.columns)}
        return df

    all_rows = []
    offset   = 0
    limit    = 1000
    page     = 1
    print(f"  ⬇️   {dataset_id} (paginated)...")

    while True:
        try:
            params   = {"id": dataset_id, "limit": limit, "offset": offset}
            response = requests.get(BASE_URL, params=params, timeout=30)
            time.sleep(REQUEST_DELAY)

            if response.status_code != 200:
                print(f"       Page {page} ❌ HTTP {response.status_code}")
                break

            data = response.json()

            if isinstance(data, dict) and "data" in data:
                rows = data["data"]
            elif isinstance(data, list):
                rows = data
            else:
                break

            if len(rows) == 0:
                break

            all_rows.extend(rows)
            print(f"       Page {page:>2} — {len(all_rows):>6,} rows so far")

            if len(rows) < limit:
                break   # Last page

            offset += limit
            page   += 1

        except requests.exceptions.Timeout:
            print(f"       Page {page} — Timeout, retrying in 5s...")
            time.sleep(5)
            continue
        except Exception as e:
            print(f"       Page {page} — Error: {e}")
            break

    if len(all_rows) == 0:
        print(f"  ❌  {dataset_id:<42} 0 rows — check dataset ID")
        download_log[dataset_id] = {"rows": 0, "status": "empty"}
        return None

    df = pd.DataFrame(all_rows)
    df.to_csv(csv_path, index=False)
    print(f"  ✅  {dataset_id:<42} {len(df):>6,} rows total  →  saved\n")
    download_log[dataset_id] = {"rows": len(df), "status": "ok", "cols": list(df.columns)}
    return df


# ============================================================
# FUNCTION 3: load_or_fetch
# Smart loader — reads CSV if exists, else downloads
# ============================================================
def load_or_fetch(dataset_id, paginated=False):
    """Auto-detect: load from CSV if exists, else download via API."""
    csv_path = f"{RAW_DIR}/{dataset_id}.csv"
    if os.path.exists(csv_path):
        return fetch_dosm(dataset_id) if not paginated else fetch_all_pages(dataset_id)
    return fetch_all_pages(dataset_id) if paginated else fetch_dosm(dataset_id)


print("✅  fetch_dosm()      — for small datasets (< 1,000 rows)")
print("✅  fetch_all_pages() — for large datasets (pagination)")
print("✅  load_or_fetch()   — smart auto-loader")
print()
print("👉  Run Cell 2 to download Tema 1 datasets.")

✅  Packages installed
✅  Folder structure ready:
     C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\data\raw
     C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\data\clean
     C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\outputs

✅  fetch_dosm()      — for small datasets (< 1,000 rows)
✅  fetch_all_pages() — for large datasets (pagination)
✅  load_or_fetch()   — smart auto-loader

👉  Run Cell 2 to download Tema 1 datasets.


In [2]:
# ============================================================
# CELL 2 — TEMA 1: The Ageing Nation Story
# Large datasets → fetch_all_pages (pagination)
# Small datasets → fetch_dosm (single request)
# ============================================================

print("=" * 60)
print("  TEMA 1 — The Ageing Nation Story")
print("  Datasets: Population, Fertility, Births, Deaths, Marriages")
print("=" * 60)
print()

# ── LARGE datasets — need pagination ────────────────────────
# population_malaysia has 5,000+ rows (age × sex × ethnicity × year)
# population_state    has 10,000+ rows
# population_district has 40,000+ rows
# fertility_state     has 2,000+ rows

print("[Large datasets — paginated download]")
pop_malaysia    = fetch_all_pages("population_malaysia")   # 1970–2024 | age × sex × ethnicity
pop_state       = fetch_all_pages("population_state")      # 1970–2024 | by state
pop_district    = fetch_all_pages("population_district")   # 2000–2024 | by district
fertility_state = fetch_all_pages("fertility_state")       # ~1980–2023 | TFR by state

print()
print("[Small datasets — single request]")
fertility  = fetch_dosm("fertility")       # 1958–2023 | National TFR
births     = fetch_dosm("births_annual")   # 2000–2023 | Live births by sex
deaths     = fetch_dosm("deaths")          # 2000–2023 | Deaths by sex
marriages  = fetch_dosm("marriages")       # 2017–2022 | Marriages by sex
hh_profile = fetch_dosm("hh_profile")      # HIES cycles | HH size, composition

print()
print("✅  Tema 1 done! Run Cell 3 for Tema 2.")

  TEMA 1 — The Ageing Nation Story
  Datasets: Population, Fertility, Births, Deaths, Marriages

[Large datasets — paginated download]
  ⏩  population_malaysia                        92,000 rows  (loaded from cache)
  ⏩  population_state                           1,807,000 rows  (loaded from cache)
  ⏩  population_district                        10,567,000 rows  (loaded from cache)
  ⏩  fertility_state                            2,280,000 rows  (loaded from cache)

[Small datasets — single request]
  ⏩  fertility                                     528 rows  (loaded from cache)
  ⏩  births_annual                                  24 rows  (loaded from cache)
  ⏩  deaths                                         24 rows  (loaded from cache)
  ⏩  marriages                                      12 rows  (loaded from cache)
  ⏩  hh_profile                                     19 rows  (loaded from cache)

✅  Tema 1 done! Run Cell 3 for Tema 2.


In [3]:
# ============================================================
# CELL 3 — TEMA 2: Income vs Inflation Story
# ============================================================

print("=" * 60)
print("  TEMA 2 — Income vs Inflation Story")
print("  Datasets: HH Income, CPI, Gini, Poverty")
print("=" * 60)
print()

# ── SMALL datasets — HIES (survey every 2-3 years) ──────────
print("[Small datasets — single request]")
hh_income           = fetch_dosm("hh_income")             # 1970–2022 | National median/mean income
hh_income_state     = fetch_dosm("hh_income_state")       # 1970–2022 | Income by state
hh_inequality       = fetch_dosm("hh_inequality")         # 1970–2022 | National Gini coefficient
hh_inequality_state = fetch_dosm("hh_inequality_state")   # 1970–2022 | Gini by state
hh_poverty          = fetch_dosm("hh_poverty")            # 1970–2022 | National poverty rates
hh_poverty_state    = fetch_dosm("hh_poverty_state")      # 1970–2022 | Poverty by state
hies_state          = fetch_dosm("hies_state")            # HIES cycles | Expenditure by state
cpi_annual          = fetch_dosm("cpi_annual")            # 1970–2024 | Annual CPI index

print()
# ── LARGE datasets — CPI monthly data (paginated) ───────────
print("[Large datasets — paginated download]")
cpi_headline  = fetch_all_pages("cpi_headline")            # 2010–2025 | Monthly CPI index
cpi_inflation = fetch_all_pages("cpi_headline_inflation")  # 2010–2025 | Monthly YoY inflation
cpi_state     = fetch_all_pages("cpi_state")               # 2010–2025 | CPI by state
cpi_lowincome = fetch_all_pages("cpi_lowincome")           # 2010–2025 | CPI for low-income HH

print()
print("✅  Tema 2 done! Run Cell 4 for summary.")

  TEMA 2 — Income vs Inflation Story
  Datasets: HH Income, CPI, Gini, Poverty

[Small datasets — single request]
  ⏩  hh_income                                      21 rows  (loaded from cache)
  ⏩  hh_income_state                               303 rows  (loaded from cache)
  ⏩  hh_inequality                                  20 rows  (loaded from cache)
  ⏩  hh_inequality_state                           273 rows  (loaded from cache)
  ⏩  hh_poverty                                     20 rows  (loaded from cache)
  ⏩  hh_poverty_state                              294 rows  (loaded from cache)
  ⏩  hies_state                                     16 rows  (loaded from cache)
  ⏩  cpi_annual                                    545 rows  (loaded from cache)

[Large datasets — paginated download]
  ⏩  cpi_headline                               4,323,000 rows  (loaded from cache)
  ⏩  cpi_headline_inflation                     1,625,000 rows  (loaded from cache)
  ⏩  cpi_state                 

In [4]:
# ============================================================
# CELL 4 — FULL DOWNLOAD SUMMARY + DATE RANGE CHECK
# ============================================================

all_datasets = {
    # ── Tema 1 ───────────────────────────────────────────────
    "population_malaysia"     : (pop_malaysia,     "Tema 1", "paginated"),
    "population_state"        : (pop_state,        "Tema 1", "paginated"),
    "population_district"     : (pop_district,     "Tema 1", "paginated"),
    "fertility"               : (fertility,        "Tema 1", "single"),
    "fertility_state"         : (fertility_state,  "Tema 1", "paginated"),
    "births_annual"           : (births,           "Tema 1", "single"),
    "deaths"                  : (deaths,           "Tema 1", "single"),
    "marriages"               : (marriages,        "Tema 1", "single"),
    "hh_profile"              : (hh_profile,       "Tema 1", "single"),
    # ── Tema 2 ───────────────────────────────────────────────
    "hh_income"               : (hh_income,            "Tema 2", "single"),
    "hh_income_state"         : (hh_income_state,       "Tema 2", "single"),
    "hh_inequality"           : (hh_inequality,         "Tema 2", "single"),
    "hh_inequality_state"     : (hh_inequality_state,   "Tema 2", "single"),
    "hh_poverty"              : (hh_poverty,            "Tema 2", "single"),
    "hh_poverty_state"        : (hh_poverty_state,      "Tema 2", "single"),
    "hies_state"              : (hies_state,            "Tema 2", "single"),
    "cpi_annual"              : (cpi_annual,            "Tema 2", "single"),
    "cpi_headline"            : (cpi_headline,          "Tema 2", "paginated"),
    "cpi_headline_inflation"  : (cpi_inflation,         "Tema 2", "paginated"),
    "cpi_state"               : (cpi_state,             "Tema 2", "paginated"),
    "cpi_lowincome"           : (cpi_lowincome,         "Tema 2", "paginated"),
}

success_list = []
failed_list  = []

print("=" * 90)
print(f"  {'DATASET':<38} {'TEMA':<8} {'ROWS':>7}  {'YEAR RANGE':<18}  COLUMNS")
print("  " + "-" * 86)

for name, (df, tema, method) in all_datasets.items():
    if df is not None and len(df) > 0:
        # Try to detect year range
        yr_range = ""
        for col in ["year", "date", "tahun"]:
            if col in df.columns:
                try:
                    yr_min = pd.to_datetime(df[col], errors='coerce').dt.year.min()
                    yr_max = pd.to_datetime(df[col], errors='coerce').dt.year.max()
                    if pd.notna(yr_min):
                        yr_range = f"{int(yr_min)}–{int(yr_max)}"
                    else:
                        yr_range = f"{df[col].min()}–{df[col].max()}"
                except:
                    yr_range = f"{df[col].min()}–{df[col].max()}"
                break

        cols_preview = ", ".join(list(df.columns)[:5])
        if len(df.columns) > 5:
            cols_preview += f" (+{len(df.columns)-5} more)"

        print(f"  ✅  {name:<36} {tema:<8} {len(df):>7,}  {yr_range:<18}  {cols_preview}")
        success_list.append(name)
    else:
        print(f"  ❌  {name:<36} {'':8} {'FAILED':>7}")
        failed_list.append(name)

print("=" * 90)
print(f"  ✅  Downloaded : {len(success_list)} / {len(all_datasets)} datasets")

if failed_list:
    print(f"  ❌  Failed     : {failed_list}")
    print()
    print("  FIX: For failed datasets, download manually:")
    for ds in failed_list:
        print(f"    → https://open.dosm.gov.my/data-catalogue/{ds}")
    print(f"    Save CSV files to: {os.path.abspath(RAW_DIR)}/")
else:
    print()
    print("  🎉  ALL 21 DATASETS DOWNLOADED SUCCESSFULLY!")
    print(f"  📁  Saved to: {os.path.abspath(RAW_DIR)}/")

print("=" * 90)
print()
print("👉  Run Cell 5 for data preview.")

  DATASET                                TEMA        ROWS  YEAR RANGE          COLUMNS
  --------------------------------------------------------------------------------------
  ✅  population_malaysia                  Tema 1    92,000  1970–1981           age, sex, date, ethnicity, population
  ✅  population_state                     Tema 1   1,807,000  2020–2020           age, sex, date, state, ethnicity (+1 more)
  ✅  population_district                  Tema 1   10,567,000  2020–2021           age, sex, date, state, district (+2 more)
  ✅  fertility                            Tema 1       528  1958–2023           date, age_group, fertility_rate
  ✅  fertility_state                      Tema 1   2,280,000  2001–2023           date, state, age_group, fertility_rate
  ✅  births_annual                        Tema 1        24  2000–2023           abs, date, rate
  ✅  deaths                               Tema 1        24  2000–2023           abs, date, rate
  ✅  marriages                 

In [5]:
# ============================================================
# CELL 5 — DATA PREVIEW: eyeball key datasets
# ============================================================

import warnings
warnings.filterwarnings('ignore')

def preview(name, df, n=3):
    if df is None:
        print(f"  ❌  {name} — not available")
        return
    print(f"\n{'─'*60}")
    print(f"  {name}  |  {df.shape[0]:,} rows × {df.shape[1]} cols")
    # Year range
    for col in ["year", "date"]:
        if col in df.columns:
            print(f"  {col} range: {df[col].min()} → {df[col].max()}")
            break
    # Unique filter values
    for col in ["sex", "ethnicity", "age", "state", "division"]:
        if col in df.columns:
            vals = df[col].unique().tolist()
            print(f"  {col}: {vals[:8]}{'...' if len(vals)>8 else ''}")
    print()
    display(df.head(n))

# ── Tema 1 Previews ─────────────────────────────────────────
print("=" * 60)
print("  TEMA 1 — KEY DATASET PREVIEWS")
print("=" * 60)

preview("population_malaysia",  pop_malaysia)
preview("fertility",            fertility)
preview("births_annual",        births)
preview("deaths",               deaths)
preview("fertility_state",      fertility_state)

# ── Tema 2 Previews ─────────────────────────────────────────
print()
print("=" * 60)
print("  TEMA 2 — KEY DATASET PREVIEWS")
print("=" * 60)

preview("hh_income",            hh_income)
preview("hh_income_state",      hh_income_state)
preview("hh_inequality",        hh_inequality)
preview("hh_inequality_state",  hh_inequality_state)
preview("hh_poverty",           hh_poverty)
preview("cpi_headline",         cpi_headline)
preview("cpi_headline_inflation", cpi_inflation)
preview("cpi_annual",           cpi_annual)
preview("cpi_lowincome",        cpi_lowincome)

  TEMA 1 — KEY DATASET PREVIEWS

────────────────────────────────────────────────────────────
  population_malaysia  |  92,000 rows × 5 cols
  date range: 1970-01-01 → 1981-01-01
  sex: ['both', 'female', 'male']
  ethnicity: ['overall', 'bumi', 'chinese', 'indian', 'other']
  age: ['overall', '0-4', '5-9', '10-14', '15-19', '20-24', '25-29', '30-34']...



,age,sex,date,ethnicity,population
0,overall,both,1970-01-01,overall,10881.8
1,0-4,both,1970-01-01,overall,1702.4
2,5-9,both,1970-01-01,overall,1690.3



────────────────────────────────────────────────────────────
  fertility  |  528 rows × 3 cols
  date range: 1958-01-01 → 2023-01-01



,date,age_group,fertility_rate
0,1958-01-01,tfr,6.28
1,1959-01-01,tfr,6.18
2,1960-01-01,tfr,6.04



────────────────────────────────────────────────────────────
  births_annual  |  24 rows × 3 cols
  date range: 2000-01-01 → 2023-01-01



...

,date,division,inflation_mom,inflation_yoy
0,2011-11-01,13,0.4,3.2
1,2011-12-01,13,0.0,3.1
2,2012-01-01,13,-0.1,2.7



────────────────────────────────────────────────────────────
  cpi_annual  |  545 rows × 3 cols
  date range: 1960-01-01 → 2024-01-01
  division: ['overall', '01', '02', '03', '04', '05', '06', '07']...



,date,index,division
0,1960-01-01,21.309241,overall
1,1961-01-01,21.270798,overall
2,1962-01-01,21.293864,overall



────────────────────────────────────────────────────────────
  cpi_lowincome  |  1,000 rows × 3 cols
  date range: 2010-01-01 → 2026-01-01
  division: ['11', 'overall', '01', '02', '03', '04', '05']



,date,index,division
0,2017-02-01,128.3,11
1,2017-03-01,128.5,11
2,2017-04-01,128.8,11


In [6]:
# ============================================================
# CELL 6 — FALLBACK: Manual download links
# Run this cell ONLY if specific datasets failed above.
# It will print direct CSV download links for each.
# ============================================================

# All known direct storage links for manual fallback
MANUAL_LINKS = {
    "population_malaysia"    : "https://storage.data.gov.my/dosm/population_malaysia.csv",
    "population_state"       : "https://storage.data.gov.my/dosm/population_state.csv",
    "population_district"    : "https://storage.data.gov.my/dosm/population_district.csv",
    "fertility"              : "https://storage.data.gov.my/dosm/fertility.csv",
    "fertility_state"        : "https://storage.data.gov.my/dosm/fertility_state.csv",
    "births_annual"          : "https://storage.data.gov.my/dosm/births_annual.csv",
    "deaths"                 : "https://storage.data.gov.my/dosm/deaths.csv",
    "marriages"              : "https://storage.data.gov.my/dosm/marriages.csv",
    "hh_profile"             : "https://storage.data.gov.my/dosm/hh_profile.csv",
    "hh_income"              : "https://storage.data.gov.my/dosm/hh_income.csv",
    "hh_income_state"        : "https://storage.data.gov.my/dosm/hh_income_state.csv",
    "hh_inequality"          : "https://storage.data.gov.my/dosm/hh_inequality.csv",
    "hh_inequality_state"    : "https://storage.data.gov.my/dosm/hh_inequality_state.csv",
    "hh_poverty"             : "https://storage.data.gov.my/dosm/hh_poverty.csv",
    "hh_poverty_state"       : "https://storage.data.gov.my/dosm/hh_poverty_state.csv",
    "hies_state"             : "https://storage.data.gov.my/dosm/hies_state.csv",
    "cpi_annual"             : "https://storage.data.gov.my/dosm/cpi_annual.csv",
    "cpi_headline"           : "https://storage.data.gov.my/dosm/cpi_headline.csv",
    "cpi_headline_inflation" : "https://storage.data.gov.my/dosm/cpi_headline_inflation.csv",
    "cpi_state"              : "https://storage.data.gov.my/dosm/cpi_state.csv",
    "cpi_lowincome"          : "https://storage.data.gov.my/dosm/cpi_lowincome.csv",
}

# Try to directly download failed datasets using storage links
if failed_list:
    print("Attempting direct CSV download for failed datasets...\n")
    for ds in failed_list:
        if ds in MANUAL_LINKS:
            url = MANUAL_LINKS[ds]
            try:
                print(f"  ⬇️   {ds}...")
                df_manual = pd.read_csv(url)
                df_manual.to_csv(f"{RAW_DIR}/{ds}.csv", index=False)
                print(f"  ✅  {ds} — {len(df_manual):,} rows downloaded via direct link")
                globals()[ds.replace("-", "_")] = df_manual
            except Exception as e:
                print(f"  ❌  {ds} — Direct link also failed: {e}")
                print(f"       Manual download: {url}")
        else:
            print(f"  ⚠️   {ds} — No direct link available")
            print(f"       Go to: https://open.dosm.gov.my/data-catalogue/{ds}")
            print(f"       Download CSV → save to: {os.path.abspath(RAW_DIR)}/{ds}.csv")
else:
    print("✅  No failed datasets — nothing to fix!")
    print()
    print("All 21 CSV files are saved in:")
    files = sorted(os.listdir(RAW_DIR))
    for f in files:
        size_kb = os.path.getsize(f"{RAW_DIR}/{f}") // 1024
        print(f"  📄  {f:<50} {size_kb:>5} KB")

✅  No failed datasets — nothing to fix!

All 21 CSV files are saved in:
  📄  .gitkeep                                               0 KB
  📄  births_annual.csv                                      0 KB
  📄  cpi_annual.csv                                        17 KB
  📄  cpi_headline.csv                                   93674 KB
  📄  cpi_headline_inflation.csv                         39214 KB
  📄  cpi_lowincome.csv                                     21 KB
  📄  cpi_state.csv                                         34 KB
  📄  deaths.csv                                             0 KB
  📄  fertility.csv                                         12 KB
  📄  fertility_state.csv                                70263 KB
  📄  hh_income.csv                                          0 KB
  📄  hh_income_state.csv                                    9 KB
  📄  hh_inequality.csv                                      0 KB
  📄  hh_inequality_state.csv                                7 KB
  📄  hh_poverty.cs

In [7]:
# ============================================================
# CELL 7 — FILE LIST + NEXT STEPS
# ============================================================

import glob

csv_files = sorted(glob.glob(f"{RAW_DIR}/*.csv"))

print("=" * 60)
print("  FILES IN ../data/raw/")
print("=" * 60)
total_kb = 0
for f in csv_files:
    size_kb  = os.path.getsize(f) // 1024
    total_kb += size_kb
    name     = os.path.basename(f)
    print(f"  📄  {name:<50} {size_kb:>5} KB")
print("─" * 60)
print(f"  {len(csv_files)} files  |  Total: {total_kb:,} KB ({total_kb//1024:.1f} MB)")
print("=" * 60)
print()
print("📋  NEXT STEPS:")
print()
print("  1. ✅  This notebook (00) — DONE")
print("  2. ▶️   Run: 01_EDA_demographics.ipynb")
print("         → Explore, clean, check null values")
print("         → Export clean CSVs to ../data/clean/")
print()
print("  3. ▶️   Run: 02_demographics_analysis.ipynb")
print("         → Build all 12+ charts (Tema 1 + 2 + Gabungan)")
print("         → Charts saved to ../outputs/")
print()
print("  4. ▶️   Run: 03_dashboard.py (Streamlit) or open Power BI")
print("         → Load clean CSVs → build interactive dashboard")
print()
print(f"  ⏰  Downloaded at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

  FILES IN ../data/raw/
  📄  births_annual.csv                                      0 KB
  📄  cpi_annual.csv                                        17 KB
  📄  cpi_headline.csv                                   93674 KB
  📄  cpi_headline_inflation.csv                         39214 KB
  📄  cpi_lowincome.csv                                     21 KB
  📄  cpi_state.csv                                         34 KB
  📄  deaths.csv                                             0 KB
  📄  fertility.csv                                         12 KB
  📄  fertility_state.csv                                70263 KB
  📄  hh_income.csv                                          0 KB
  📄  hh_income_state.csv                                    9 KB
  📄  hh_inequality.csv                                      0 KB
  📄  hh_inequality_state.csv                                7 KB
  📄  hh_poverty.csv                                         0 KB
  📄  hh_poverty_state.csv                                   9 KB
 